# NumCompute-Stream — Streaming Demo
**Assignment 2.2 | Programming for AI | Adelaide University**

This notebook demonstrates the full streaming pipeline:
1. Load data from CSV via `io.py`
2. Split into chunks to simulate a stream
3. Train incrementally with `partial_fit()`
4. Log and visualise metrics using `visualise.py`
5. Compare single tree vs ensemble methods

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from numcompute_stream.io import load_csv_with_header
from numcompute_stream.preprocessing import StandardScaler, SimpleImputer
from numcompute_stream.ensemble import EnsembleClassifier
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.metrics import StreamingMetrics
from numcompute_stream.stats import StreamingStats
from numcompute_stream import visualise

print('All imports OK')

All imports OK


## 1. Load CSV Data

In [2]:
headers, data = load_csv_with_header('../data/health_stream.csv')
print('Columns:', headers)
print('Shape:  ', data.shape)
print('Sample:\n', data[:5])

Columns: ['age', 'height', 'weight', 'bmi', 'label']
Shape:   (300, 5)
Sample:
 [[ 24.   166.    86.    31.21   1.  ]
 [ 54.     0.    96.    27.45   1.  ]
 [ 49.   150.    94.    41.78   1.  ]
 [ 39.   190.    58.    16.07   0.  ]
 [ 39.   174.    99.    32.7    1.  ]]


## 2. Prepare Features & Labels — Split into Chunks

In [3]:
X = data[:, :-1]          # age, height, weight, bmi
y = data[:, -1].astype(int)   # label (0/1)

CHUNK_SIZE = 30
n = len(X)
chunks = [(X[i:i+CHUNK_SIZE], y[i:i+CHUNK_SIZE])
          for i in range(0, n, CHUNK_SIZE)]

print(f'Total samples: {n}  |  Chunks: {len(chunks)}  |  Chunk size: {CHUNK_SIZE}')

Total samples: 300  |  Chunks: 10  |  Chunk size: 30


## 3. Build Streaming Pipelines

In [4]:
def make_rf_pipe():
    return Pipeline([
        ('impute', SimpleImputer(fill_value=0.0)),
        ('scale',  StandardScaler()),
        ('model',  EnsembleClassifier(method='random_forest',
                                       n_estimators=10, max_depth=4)),
    ])

def make_tree_pipe():
    return Pipeline([
        ('impute', SimpleImputer(fill_value=0.0)),
        ('scale',  StandardScaler()),
        ('model',  DecisionTreeClassifier(max_depth=4)),
    ])

def make_ada_pipe():
    return Pipeline([
        ('impute', SimpleImputer(fill_value=0.0)),
        ('scale',  StandardScaler()),
        ('model',  EnsembleClassifier(method='adaboost',
                                       n_estimators=10, max_depth=1)),
    ])

print('Pipelines ready')

Pipelines ready


## 4. Train Incrementally — Chunk by Chunk

In [5]:
print('=== Random Forest ===')
trainer_rf = StreamTrainer(make_rf_pipe(), verbose=True)
trainer_rf.run(chunks)
trainer_rf.print_summary()

print('\n=== Single Decision Tree ===')
trainer_tree = StreamTrainer(make_tree_pipe(), verbose=True)
trainer_tree.run(chunks)
trainer_tree.print_summary()

print('\n=== AdaBoost ===')
trainer_ada = StreamTrainer(make_ada_pipe(), verbose=True)
trainer_ada.run(chunks)
trainer_ada.print_summary()

=== Random Forest ===
  Chunk   0 | n=   30 | acc=1.0000 | cum_acc=1.0000 | fit=10.5ms
  Chunk   1 | n=   30 | acc=1.0000 | cum_acc=1.0000 | fit=26.4ms
  Chunk   2 | n=   30 | acc=1.0000 | cum_acc=1.0000 | fit=40.6ms
  Chunk   3 | n=   30 | acc=1.0000 | cum_acc=1.0000 | fit=62.0ms
  Chunk   4 | n=   30 | acc=1.0000 | cum_acc=1.0000 | fit=71.9ms
  Chunk   5 | n=   30 | acc=1.0000 | cum_acc=1.0000 | fit=61.0ms
  Chunk   6 | n=   30 | acc=1.0000 | cum_acc=1.0000 | fit=82.7ms
  Chunk   7 | n=   30 | acc=0.9667 | cum_acc=0.9958 | fit=94.7ms
  Chunk   8 | n=   30 | acc=1.0000 | cum_acc=0.9963 | fit=123.0ms
  Chunk   9 | n=   30 | acc=1.0000 | cum_acc=0.9967 | fit=167.0ms

  StreamTrainer Summary
  Chunks processed  : 10
  Total samples     : 300
  Final cum. acc.   : 0.9967
  Total fit time    : 739.9 ms

=== Single Decision Tree ===
  Chunk   0 | n=   30 | acc=1.0000 | cum_acc=1.0000 | fit=2.6ms
  Chunk   1 | n=   30 | acc=1.0000 | cum_acc=1.0000 | fit=4.6ms
  Chunk   2 | n=   30 | acc=1.00

## 5. Visualise — Single Model Accuracy

In [6]:
fig = visualise.plot_metric_over_time(
    trainer_rf.get_metric_history('accuracy'),
    title='Random Forest — Chunk Accuracy over Stream',
    ylabel='Accuracy', show=False
)
fig.savefig('rf_accuracy.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved rf_accuracy.png')

Saved rf_accuracy.png


C:\Users\heman\AppData\Local\Temp\ipykernel_29524\3388520344.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Compare Models

In [7]:
fig = visualise.compare_models(
    trainer_rf.get_metric_history('cumulative_acc'),
    trainer_tree.get_metric_history('cumulative_acc'),
    labels=['Random Forest', 'Single Tree'],
    title='Cumulative Accuracy: Ensemble vs Single Tree',
    ylabel='Cumulative Accuracy', show=False
)
fig.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved model_comparison.png')

Saved model_comparison.png


C:\Users\heman\AppData\Local\Temp\ipykernel_29524\385329311.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Predictions vs Ground Truth (Last Chunk)

In [8]:
X_last, y_last = chunks[-1]
y_pred_last = trainer_rf.model.predict(X_last)

fig = visualise.plot_predictions_vs_ground_truth(
    y_last, y_pred_last,
    title='Last Chunk: RF Predictions vs Truth',
    show=False
)
fig.savefig('predictions_vs_truth.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved predictions_vs_truth.png')

Saved predictions_vs_truth.png


C:\Users\heman\AppData\Local\Temp\ipykernel_29524\903346847.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Streaming Metrics — Confusion Matrix

In [9]:
sm = StreamingMetrics(n_classes=2)
for X_c, y_c in chunks:
    y_pred = trainer_rf.model.predict(X_c)
    sm.update(y_c, y_pred)

result = sm.result()
print(f"Accuracy : {result['accuracy']:.4f}")
print(f"Precision: {result['precision']:.4f}")
print(f"Recall   : {result['recall']:.4f}")
print(f"F1       : {result['f1']:.4f}")

fig = visualise.plot_confusion_matrix(
    result['confusion_matrix'],
    class_names=['Normal', 'Overweight'],
    title='Cumulative Confusion Matrix — Random Forest',
    show=False
)
fig.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved confusion_matrix.png')

Accuracy : 0.9933
Precision: 0.9945
Recall   : 0.9916
F1       : 0.9930
Saved confusion_matrix.png


C:\Users\heman\AppData\Local\Temp\ipykernel_29524\4088769311.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Streaming Statistics

In [10]:
ss = StreamingStats(window_size=90)
for X_c, _ in chunks:
    ss.update_stats(X_c)

summary = ss.summary()
print('Running stats over full stream:')
for k, v in summary.items():
    print(f'  {k}: {v}')

Running stats over full stream:
  mean: [ 41.97666667 163.56333333  80.35        27.74463333]
  std: [12.2579004  39.80158274 17.76271845  7.92097828]
  min: [20.    0.   50.   13.82]
  max: [ 64.   194.   109.    46.67]
  n_seen: 300


## 10. Full Dashboard

In [11]:
fig = visualise.plot_streaming_dashboard(
    chunk_accuracies     = trainer_rf.get_metric_history('accuracy'),
    cumulative_accuracies= trainer_rf.get_metric_history('cumulative_acc'),
    fit_times            = trainer_rf.get_metric_history('fit_time_s'),
    y_true_last          = y_last,
    y_pred_last          = y_pred_last,
    cm                   = result['confusion_matrix'],
    model_name           = 'Random Forest',
    show=False
)
fig.savefig('dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print('Dashboard saved as dashboard.png')

c:\Users\heman\Downloads\numcompute_stream_fixed\numcompute_stream\demo\..\numcompute_stream\visualise.py:86: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
c:\Users\heman\Downloads\numcompute_stream_fixed\numcompute_stream\demo\..\numcompute_stream\visualise.py:189: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
c:\Users\heman\Downloads\numcompute_stream_fixed\numcompute_stream\demo\..\numcompute_stream\visualise.py:241: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
c:\Users\heman\Downloads\numcompute_stream_fixed\numcompute_stream\demo\..\numcompute_stream\visualise.py:363: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


Dashboard saved as dashboard.png


C:\Users\heman\AppData\Local\Temp\ipykernel_29524\1848169449.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
